# m03 · Refrigeration cycles and aircraft air conditioning: verification calculations

This notebook recomputes in Python the main examples of module m03 (Slide Ch.3; Tutorials I; Cengel Ch.11). Every section has three parts: the question the method answers → code → interpretation of the real printed numbers. Run top to bottom. The R-134a tables (A-11, A-12, the A-13 blocks) and Table A-17 are embedded in the first code cell; if CoolProp is installed each section also compares with CoolProp (the independent route), otherwise that part is skipped. The cooling-pack part uses only slide numbers (illustration); no book describes this system.

## 1. COP of a reversed-Carnot refrigerator

**Question:** how does COP_R,Carnot = 1/(T_H/T_L − 1) change as the two temperatures come closer? Try T_H = 35 °C and several T_L (slide Ch.3 p.13).

In [1]:
import math, numpy as np
from scipy.integrate import solve_ivp
try:
    import CoolProp.CoolProp as CP
    CP.set_reference_state('R134a', 'ASHRAE')          # h = s = 0 for saturated liquid at -40 C, as in the tables
    HAVE_CP = True
except Exception:
    HAVE_CP = False
# R-134a, Appendix 1 tables read from the page images. A-11: T, Psat, hf, hfg, hg, sf, sfg, sg ; A-12: Psat, Tsat, hf, hfg, hg, sf, sfg, sg
A11 = np.array([[-40.0, 51.25, 0.0, 225.86, 225.86, 0.0, 0.96866, 0.96866], [-38.0, 56.86, 2.515, 224.61, 227.12, 0.01072, 0.95511, 0.96584], [-36.0, 62.95, 5.037, 223.35, 228.39, 0.02138, 0.94176, 0.96315], [-34.0, 69.56, 7.566, 222.09, 229.65, 0.03199, 0.92859, 0.96058], [-32.0, 76.71, 10.1, 220.81, 230.91, 0.04253, 0.9156, 0.95813], [-30.0, 84.43, 12.65, 219.52, 232.17, 0.05301, 0.90278, 0.95579], [-28.0, 92.76, 15.2, 218.22, 233.43, 0.06344, 0.89012, 0.95356], [-26.0, 101.73, 17.76, 216.92, 234.68, 0.07382, 0.87762, 0.95144], [-24.0, 111.37, 20.33, 215.59, 235.92, 0.08414, 0.86527, 0.94941], [-22.0, 121.72, 22.91, 214.26, 237.17, 0.09441, 0.85307, 0.94748], [-20.0, 132.82, 25.49, 212.91, 238.41, 0.10463, 0.84101, 0.94564], [-18.0, 144.69, 28.09, 211.55, 239.64, 0.11481, 0.82908, 0.94389], [-16.0, 157.38, 30.69, 210.18, 240.87, 0.12493, 0.81729, 0.94222], [-14.0, 170.93, 33.3, 208.79, 242.09, 0.13501, 0.80561, 0.94063], [-12.0, 185.37, 35.92, 207.38, 243.3, 0.14504, 0.79406, 0.93911], [-10.0, 200.74, 38.55, 205.96, 244.51, 0.15504, 0.78263, 0.93766], [-8.0, 217.08, 41.19, 204.52, 245.72, 0.16498, 0.7713, 0.93629], [-6.0, 234.44, 43.84, 203.07, 246.91, 0.17489, 0.76008, 0.93497], [-4.0, 252.85, 46.5, 201.6, 248.1, 0.18476, 0.74896, 0.93372], [-2.0, 272.36, 49.17, 200.11, 249.28, 0.19459, 0.73794, 0.93253], [0.0, 293.01, 51.86, 198.6, 250.45, 0.20439, 0.72701, 0.93139], [2.0, 314.84, 54.55, 197.07, 251.61, 0.21415, 0.71616, 0.93031], [4.0, 337.9, 57.25, 195.51, 252.77, 0.22387, 0.7054, 0.92927], [6.0, 362.23, 59.97, 193.94, 253.91, 0.23356, 0.69471, 0.92828], [8.0, 387.88, 62.69, 192.35, 255.04, 0.24323, 0.6841, 0.92733], [10.0, 414.89, 65.43, 190.73, 256.16, 0.25286, 0.67356, 0.92641], [12.0, 443.31, 68.18, 189.09, 257.27, 0.26246, 0.66308, 0.92554], [14.0, 473.19, 70.95, 187.42, 258.37, 0.27204, 0.65266, 0.9247], [16.0, 504.58, 73.73, 185.73, 259.46, 0.28159, 0.6423, 0.92389], [18.0, 537.52, 76.52, 184.01, 260.53, 0.29112, 0.63198, 0.9231], [20.0, 572.07, 79.32, 182.27, 261.59, 0.30063, 0.62172, 0.92234], [22.0, 608.27, 82.14, 180.49, 262.64, 0.31011, 0.61149, 0.9216], [24.0, 646.18, 84.98, 178.69, 263.67, 0.31958, 0.6013, 0.92088], [26.0, 685.84, 87.83, 176.85, 264.68, 0.32903, 0.59115, 0.92018], [28.0, 727.31, 90.69, 174.99, 265.68, 0.33846, 0.58102, 0.91948], [30.0, 770.64, 93.58, 173.08, 266.66, 0.34789, 0.57091, 0.91879], [32.0, 815.89, 96.48, 171.14, 267.62, 0.3573, 0.56082, 0.91811], [34.0, 863.11, 99.4, 169.17, 268.57, 0.3667, 0.55074, 0.91743], [36.0, 912.35, 102.33, 167.16, 269.49, 0.37609, 0.54066, 0.91675], [38.0, 963.68, 105.29, 165.1, 270.39, 0.38548, 0.53058, 0.91606], [40.0, 1017.1, 108.26, 163.0, 271.27, 0.39486, 0.52049, 0.91536], [42.0, 1072.8, 111.26, 160.86, 272.12, 0.40425, 0.51039, 0.91464], [44.0, 1130.7, 114.28, 158.67, 272.95, 0.41363, 0.50027, 0.91391], [46.0, 1191.0, 117.32, 156.43, 273.75, 0.42302, 0.49012, 0.91315], [48.0, 1253.6, 120.39, 154.14, 274.53, 0.43242, 0.47993, 0.91236], [52.0, 1386.2, 126.59, 149.39, 275.98, 0.45126, 0.45941, 0.91067], [56.0, 1529.1, 132.91, 144.38, 277.3, 0.47018, 0.43863, 0.9088], [60.0, 1682.8, 139.36, 139.1, 278.46, 0.4892, 0.41749, 0.90669], [65.0, 1891.0, 147.62, 132.02, 279.64, 0.5132, 0.39039, 0.90359], [70.0, 2118.2, 156.13, 124.32, 280.46, 0.53755, 0.36227, 0.89982], [75.0, 2365.8, 164.98, 115.85, 280.82, 0.56241, 0.33272, 0.89512], [80.0, 2635.3, 174.24, 106.35, 280.59, 0.588, 0.30111, 0.88912], [85.0, 2928.2, 184.07, 95.44, 279.51, 0.61473, 0.26644, 0.88117], [90.0, 3246.9, 194.76, 82.35, 277.11, 0.64336, 0.22674, 0.8701], [95.0, 3594.1, 207.05, 65.21, 272.26, 0.67578, 0.17711, 0.85289], [100.0, 3975.1, 224.79, 33.58, 258.37, 0.72217, 0.08999, 0.81215]])
A12 = np.array([[60.0, -36.95, 3.841, 223.95, 227.79, 0.01634, 0.94807, 0.96441], [70.0, -33.87, 7.73, 222.0, 229.73, 0.03267, 0.92775, 0.96042], [80.0, -31.13, 11.21, 220.25, 231.46, 0.04711, 0.90999, 0.9571], [90.0, -28.65, 14.37, 218.65, 233.02, 0.06008, 0.89419, 0.95427], [100.0, -26.37, 17.28, 217.16, 234.44, 0.07188, 0.87995, 0.95183], [120.0, -22.32, 22.49, 214.48, 236.97, 0.09275, 0.85503, 0.94779], [140.0, -18.77, 27.08, 212.08, 239.16, 0.11087, 0.83368, 0.94456], [160.0, -15.6, 31.21, 209.9, 241.11, 0.12693, 0.81496, 0.9419], [180.0, -12.73, 34.97, 207.9, 242.86, 0.14139, 0.79826, 0.93965], [200.0, -10.09, 38.43, 206.03, 244.46, 0.15457, 0.78316, 0.93773], [240.0, -5.38, 44.66, 202.62, 247.28, 0.17794, 0.75664, 0.93458], [280.0, -1.25, 50.18, 199.54, 249.72, 0.19829, 0.73381, 0.9321], [320.0, 2.46, 55.16, 196.71, 251.88, 0.21637, 0.71369, 0.93006], [360.0, 5.82, 59.72, 194.08, 253.81, 0.2327, 0.69566, 0.92836], [400.0, 8.91, 63.94, 191.62, 255.55, 0.24761, 0.67929, 0.92691], [450.0, 12.46, 68.81, 188.71, 257.53, 0.26465, 0.66069, 0.92535], [500.0, 15.71, 73.33, 185.98, 259.3, 0.28023, 0.64377, 0.924], [550.0, 18.73, 77.54, 183.38, 260.92, 0.29461, 0.62821, 0.92282], [600.0, 21.55, 81.51, 180.9, 262.4, 0.30799, 0.61378, 0.92177], [650.0, 24.2, 85.26, 178.51, 263.77, 0.32051, 0.6003, 0.92081], [700.0, 26.69, 88.82, 176.21, 265.03, 0.3323, 0.58763, 0.91994], [750.0, 29.06, 92.22, 173.98, 266.2, 0.34345, 0.57567, 0.91912], [800.0, 31.31, 95.47, 171.82, 267.29, 0.35404, 0.56431, 0.91835], [850.0, 33.45, 98.6, 169.71, 268.31, 0.36413, 0.55349, 0.91762], [900.0, 35.51, 101.61, 167.66, 269.26, 0.37377, 0.54315, 0.91692], [950.0, 37.48, 104.51, 165.64, 270.15, 0.38301, 0.53323, 0.91624], [1000.0, 39.37, 107.32, 163.67, 270.99, 0.39189, 0.52368, 0.91558], [1200.0, 46.29, 117.77, 156.1, 273.87, 0.42441, 0.48863, 0.91303], [1400.0, 52.4, 127.22, 148.9, 276.12, 0.45315, 0.45734, 0.9105], [1600.0, 57.88, 135.93, 141.93, 277.86, 0.47911, 0.42873, 0.90784], [1800.0, 62.87, 144.07, 135.11, 279.17, 0.50294, 0.40204, 0.90498], [2000.0, 67.45, 151.76, 128.33, 280.09, 0.52509, 0.37675, 0.90184], [2500.0, 77.54, 169.63, 111.16, 280.79, 0.57531, 0.31695, 0.89226], [3000.0, 86.16, 186.46, 92.63, 279.09, 0.62118, 0.25776, 0.87894]])
# A-13 superheated R-134a (blocks used here): P_MPa -> dict(sat=[v, u, h, s], rows=[[T, v, u, h, s], ...])
A13 = {0.14: {'sat': [0.14014, 219.54, 239.16, 0.9446], 'rows': [[-10.0, 0.14605, 225.91, 246.36, 0.9724], [0.0, 0.15263, 233.23, 254.6, 1.0031], [10.0, 0.15908, 240.66, 262.93, 1.0331], [20.0, 0.16544, 248.22, 271.38, 1.0624], [30.0, 0.17172, 255.93, 279.97, 1.0912], [40.0, 0.17794, 263.79, 288.7, 1.1195], [50.0, 0.18412, 271.79, 297.57, 1.1474], [60.0, 0.19025, 279.96, 306.59, 1.1749], [70.0, 0.19635, 288.28, 315.77, 1.202], [80.0, 0.20242, 296.75, 325.09, 1.2288], [90.0, 0.20847, 305.38, 334.57, 1.2553], [100.0, 0.21449, 314.17, 344.2, 1.2814]]}, 0.4: {'sat': [0.051201, 235.07, 255.55, 0.9269], 'rows': [[10.0, 0.051506, 235.97, 256.58, 0.9305], [20.0, 0.054213, 244.18, 265.86, 0.9628], [30.0, 0.056796, 252.36, 275.07, 0.9937], [40.0, 0.059292, 260.58, 284.3, 1.0236], [50.0, 0.061724, 268.9, 293.59, 1.0528], [60.0, 0.064104, 277.32, 302.96, 1.0814], [70.0, 0.066443, 285.86, 312.44, 1.1094], [80.0, 0.068747, 294.53, 322.02, 1.1369], [90.0, 0.071023, 303.32, 331.73, 1.164], [100.0, 0.073274, 312.26, 341.57, 1.1907], [110.0, 0.075504, 321.33, 351.53, 1.2171], [120.0, 0.077717, 330.55, 361.63, 1.2431], [130.0, 0.079913, 339.9, 371.87, 1.2688], [140.0, 0.082096, 349.41, 382.24, 1.2942]]}, 0.7: {'sat': [0.029361, 244.48, 265.03, 0.9199], 'rows': [[30.0, 0.029966, 247.48, 268.45, 0.9313], [40.0, 0.031696, 256.39, 278.57, 0.9641], [50.0, 0.033322, 265.2, 288.53, 0.9954], [60.0, 0.034875, 274.01, 298.42, 1.0256], [70.0, 0.036373, 282.87, 308.33, 1.0549], [80.0, 0.037829, 291.8, 318.28, 1.0835], [90.0, 0.03925, 300.82, 328.29, 1.1114], [100.0, 0.040642, 309.95, 338.4, 1.1389], [110.0, 0.04201, 319.19, 348.6, 1.1658], [120.0, 0.043358, 328.55, 358.9, 1.1924], [130.0, 0.044688, 338.04, 369.32, 1.2186], [140.0, 0.046004, 347.66, 379.86, 1.2444], [150.0, 0.047306, 357.41, 390.52, 1.2699], [160.0, 0.048597, 367.29, 401.31, 1.2951]]}, 0.8: {'sat': [0.025621, 246.79, 267.29, 0.9183], 'rows': [[40.0, 0.027035, 254.82, 276.45, 0.948], [50.0, 0.028547, 263.86, 286.69, 0.9802], [60.0, 0.029973, 272.83, 296.81, 1.011], [70.0, 0.03134, 281.81, 306.88, 1.0408], [80.0, 0.032659, 290.84, 316.97, 1.0698], [90.0, 0.033941, 299.95, 327.1, 1.0981], [100.0, 0.035193, 309.15, 337.3, 1.1258], [110.0, 0.03642, 318.45, 347.59, 1.153], [120.0, 0.037625, 327.87, 357.97, 1.1798], [130.0, 0.038813, 337.4, 368.45, 1.2061], [140.0, 0.039985, 347.06, 379.05, 1.2321], [150.0, 0.041143, 356.85, 389.76, 1.2577], [160.0, 0.04229, 366.76, 400.59, 1.283], [170.0, 0.043427, 376.81, 411.55, 1.308], [180.0, 0.044554, 386.99, 422.64, 1.3327]]}, 1.0: {'sat': [0.020313, 250.68, 270.99, 0.9156], 'rows': [[40.0, 0.020406, 251.3, 271.71, 0.9179], [50.0, 0.021796, 260.94, 282.74, 0.9525], [60.0, 0.023068, 270.32, 293.38, 0.985], [70.0, 0.024261, 279.59, 303.85, 1.016], [80.0, 0.025398, 288.86, 314.25, 1.0458], [90.0, 0.026492, 298.15, 324.64, 1.0748], [100.0, 0.027552, 307.51, 335.06, 1.1031], [110.0, 0.028584, 316.94, 345.53, 1.1308], [120.0, 0.029592, 326.47, 356.06, 1.158], [130.0, 0.030581, 336.11, 366.69, 1.1846], [140.0, 0.031554, 345.85, 377.4, 1.2109], [150.0, 0.032512, 355.71, 388.22, 1.2368], [160.0, 0.033457, 365.7, 399.15, 1.2623], [170.0, 0.034392, 375.81, 410.2, 1.2875], [180.0, 0.035317, 386.04, 421.36, 1.3124]]}}
# Table A-17 (air, ideal gas), rows up to 700 K: T [K], h [kJ/kg], P_r, u, v_r, s0
A17 = np.array([[200.0, 199.97, 0.3363, 142.56, 1707.0, 1.29559], [210.0, 209.97, 0.3987, 149.69, 1512.0, 1.34444], [220.0, 219.97, 0.469, 156.82, 1346.0, 1.39105], [230.0, 230.02, 0.5477, 164.0, 1205.0, 1.43557], [240.0, 240.02, 0.6355, 171.13, 1084.0, 1.47824], [250.0, 250.05, 0.7329, 178.28, 979.0, 1.51917], [260.0, 260.09, 0.8405, 185.45, 887.8, 1.55848], [270.0, 270.11, 0.959, 192.6, 808.0, 1.59634], [280.0, 280.13, 1.0889, 199.75, 738.0, 1.63279], [285.0, 285.14, 1.1584, 203.33, 706.1, 1.65055], [290.0, 290.16, 1.2311, 206.91, 676.1, 1.66802], [295.0, 295.17, 1.3068, 210.49, 647.9, 1.68515], [298.0, 298.18, 1.3543, 212.64, 631.9, 1.69528], [300.0, 300.19, 1.386, 214.07, 621.2, 1.70203], [305.0, 305.22, 1.4686, 217.67, 596.0, 1.71865], [310.0, 310.24, 1.5546, 221.25, 572.3, 1.73498], [315.0, 315.27, 1.6442, 224.85, 549.8, 1.75106], [320.0, 320.29, 1.7375, 228.42, 528.6, 1.7669], [325.0, 325.31, 1.8345, 232.02, 508.4, 1.78249], [330.0, 330.34, 1.9352, 235.61, 489.4, 1.79783], [340.0, 340.42, 2.149, 242.82, 454.1, 1.8279], [350.0, 350.49, 2.379, 250.02, 422.2, 1.85708], [360.0, 360.58, 2.626, 257.24, 393.4, 1.88543], [370.0, 370.67, 2.892, 264.46, 367.2, 1.91313], [380.0, 380.77, 3.176, 271.69, 343.4, 1.94001], [390.0, 390.88, 3.481, 278.93, 321.5, 1.96633], [400.0, 400.98, 3.806, 286.16, 301.6, 1.99194], [410.0, 411.12, 4.153, 293.43, 283.3, 2.01699], [420.0, 421.26, 4.522, 300.69, 266.6, 2.04142], [430.0, 431.43, 4.915, 307.99, 251.1, 2.06533], [440.0, 441.61, 5.332, 315.3, 236.8, 2.0887], [450.0, 451.8, 5.775, 322.62, 223.6, 2.11161], [460.0, 462.02, 6.245, 329.97, 211.4, 2.13407], [470.0, 472.24, 6.742, 337.32, 200.1, 2.15604], [480.0, 482.49, 7.268, 344.7, 189.5, 2.1776], [490.0, 492.74, 7.824, 352.08, 179.7, 2.19876], [500.0, 503.02, 8.411, 359.49, 170.6, 2.21952], [510.0, 513.32, 9.031, 366.92, 162.1, 2.23993], [520.0, 523.63, 9.684, 374.36, 154.1, 2.25997], [530.0, 533.98, 10.37, 381.84, 146.7, 2.27967], [540.0, 544.35, 11.1, 389.34, 139.7, 2.29906], [550.0, 555.74, 11.86, 396.86, 133.1, 2.31809], [560.0, 565.17, 12.66, 404.42, 127.0, 2.33685], [570.0, 575.59, 13.5, 411.97, 121.2, 2.35531], [580.0, 586.04, 14.38, 419.55, 115.7, 2.37348], [590.0, 596.52, 15.31, 427.15, 110.6, 2.3914], [600.0, 607.02, 16.28, 434.78, 105.8, 2.40902], [610.0, 617.53, 17.3, 442.42, 101.2, 2.42644], [620.0, 628.07, 18.36, 450.09, 96.92, 2.44356], [630.0, 638.63, 19.84, 457.78, 92.84, 2.46048], [640.0, 649.22, 20.64, 465.5, 88.99, 2.47716], [650.0, 659.84, 21.86, 473.25, 85.34, 2.49364], [660.0, 670.47, 23.13, 481.01, 81.89, 2.50985], [670.0, 681.14, 24.46, 488.81, 78.61, 2.52589], [680.0, 691.82, 25.85, 496.62, 75.5, 2.54175], [690.0, 702.52, 27.29, 504.45, 72.56, 2.55731], [700.0, 713.27, 28.8, 512.33, 69.76, 2.57277]])
def satT(T, col):                                   # A-11 by temperature (linear between rows)
    return float(np.interp(T, A11[:, 0], A11[:, col]))
def satP(P, col):                                   # A-12 by pressure [kPa]
    return float(np.interp(P, A12[:, 0], A12[:, col]))
HF, HFG, HG, SF, SFG, SG = 2, 3, 4, 5, 6, 7        # column numbers (same in A-11 and A-12 after the first two columns)
def A12col(P, name): return satP(P, dict(Tsat=1, hf=2, hfg=3, hg=4, sf=5, sfg=6, sg=7)[name])
def A11col(T, name): return satT(T, dict(Psat=1, hf=2, hfg=3, hg=4, sf=5, sfg=6, sg=7)[name])
def h_at_s(P_MPa, s):                                # h and T at (P, s): linear interpolation in s inside the A-13 block (the book's method)
    blk = A13[P_MPa]; rows = [[None] + blk['sat']] + blk['rows']          # saturated row first: [T, v, u, h, s]
    S = [r[4] for r in rows]; H = [r[3] for r in rows]
    for i in range(len(S) - 1):
        if S[i] <= s <= S[i + 1]:
            t = (s - S[i]) / (S[i + 1] - S[i]); return H[i] + t * (H[i + 1] - H[i])
    raise ValueError('s outside the block')
def sh(P_MPa, T, col):                               # exact grid point of an A-13 block: col in v, u, h, s
    for r in A13[P_MPa]['rows']:
        if r[0] == T: return r[dict(v=1, u=2, h=3, s=4)[col]]
    raise ValueError('not a grid point')
def tab(col, T): return float(np.interp(T, A17[:, 0], A17[:, dict(T=0, h=1, Pr=2, u=3, vr=4, s0=5)[col]]))
def inv(col, val):
    x = A17[:, dict(T=0, h=1, Pr=2, u=3, vr=4, s0=5)[col]]; return float(np.interp(val, x, A17[:, 0]))
def carnot_R(TL, TH): return 1.0 / (TH / TL - 1.0)
def carnot_HP(TL, TH): return 1.0 / (1.0 - TL / TH)
TH = 35 + 273.15
for TLc in (-20, -10, 0, 10):
    TL = TLc + 273.15
    print(f"TL = {TLc:4d} C, TH = 35 C: COP_R = {carnot_R(TL, TH):5.2f}, COP_HP = {carnot_HP(TL, TH):5.2f} (COP_HP - COP_R = {carnot_HP(TL, TH) - carnot_R(TL, TH):.2f})")

TL =  -20 C, TH = 35 C: COP_R =  4.60, COP_HP =  5.60 (COP_HP - COP_R = 1.00)
TL =  -10 C, TH = 35 C: COP_R =  5.85, COP_HP =  6.85 (COP_HP - COP_R = 1.00)
TL =    0 C, TH = 35 C: COP_R =  7.80, COP_HP =  8.80 (COP_HP - COP_R = 1.00)
TL =   10 C, TH = 35 C: COP_R = 11.33, COP_HP = 12.33 (COP_HP - COP_R = 1.00)


**Reading the output:** COP_R rises from 4.60 to 11.33 as T_L rises from −20 to 10 °C; the difference COP_HP − COP_R is always 1.00 (Cengel Eq. 11–3). Temperatures must be in kelvin: °C gives nonsense.

## 2. Tutorials I: ideal R-134a vapor-compression cycle 0.14–0.8 MPa from the tables

**Question:** compute h₁…h₄, Q̇_L, Ẇ_in, Q̇_H and COP from the tables (interpolation in s) and compare with the slide answers (239.16; 275.39; 95.47; 7.18 kW; 1.81 kW; COP 3.97) and with CoolProp.

In [2]:
def ideal(Pl, Ph):                                   # kPa: evaporator and condenser pressures
    h1, s1 = A12col(Pl, 'hg'), A12col(Pl, 'sg'); h3 = A12col(Ph, 'hf'); h2 = h_at_s(Ph / 1000, s1); h4 = h3
    x4 = (h4 - A12col(Pl, 'hf')) / A12col(Pl, 'hfg')
    return dict(h1=h1, h2=h2, h3=h3, h4=h4, s1=s1, qL=h1 - h4, w=h2 - h1, qH=h2 - h3, cop=(h1 - h4) / (h2 - h1), x4=x4)
def ideal_cp(Pl, Ph):                                # independent route: CoolProp R134a
    h1 = CP.PropsSI('H', 'P', Pl*1e3, 'Q', 1, 'R134a')/1e3; s1 = CP.PropsSI('S', 'P', Pl*1e3, 'Q', 1, 'R134a')
    h2 = CP.PropsSI('H', 'P', Ph*1e3, 'S', s1, 'R134a')/1e3; h3 = CP.PropsSI('H', 'P', Ph*1e3, 'Q', 0, 'R134a')/1e3
    return dict(h1=h1, h2=h2, h3=h3, cop=(h1-h3)/(h2-h1))
m = 0.05; c = ideal(140, 800)
print(f"h1 = {c['h1']:.2f}, h2 = {c['h2']:.2f}, h3 = h4 = {c['h3']:.2f} kJ/kg, x4 = {c['x4']:.3f}")
print(f"QL = {m*c['qL']:.2f} kW, Win = {m*c['w']:.2f} kW, QH = {m*c['qH']:.2f} kW, COP = {c['cop']:.3f}")
if HAVE_CP:
    k = ideal_cp(140, 800); print(f"CoolProp: h1 = {k['h1']:.3f}, h2 = {k['h2']:.3f}, h3 = {k['h3']:.3f}, COP = {k['cop']:.3f}")

h1 = 239.16, h2 = 275.39, h3 = h4 = 95.47 kJ/kg, x4 = 0.322
QL = 7.18 kW, Win = 1.81 kW, QH = 9.00 kW, COP = 3.966
CoolProp: h1 = 239.176, h2 = 275.388, h3 = 95.501, COP = 3.968


**Reading the output:** the tables give h₁ = 239.16, h₂ = 275.39, h₃ = 95.47; Q̇_L = 7.18 kW, Ẇ_in = 1.81 kW, COP = 3.966: the numbers of slides p.25–27. CoolProp differs by ≤ 0.03 kJ/kg in enthalpy and 0.002 in COP. x₄ = 0.322: about one third of the refrigerant flashes to vapor in the valve.

## 3. Test the method on Cengel Problem 11–12 (printed answer)

**Question:** does the same `ideal` function give 7.41 kW; 1.83 kW; 9.23 kW; 4.06 for the 0.12–0.7 MPa pair at 0.05 kg/s?

In [3]:
c12 = ideal(120, 700)
print(f"QL = {m*c12['qL']:.2f} kW, Win = {m*c12['w']:.2f} kW, QH = {m*c12['qH']:.3f} kW, COP = {c12['cop']:.3f}")
print('printed: 7.41, 1.83, 9.23, 4.06')
if HAVE_CP:
    k = ideal_cp(120, 700); print(f"CoolProp: QH = {m*(k['h2']-k['h3']):.3f} kW, COP = {k['cop']:.3f}")

QL = 7.41 kW, Win = 1.83 kW, QH = 9.236 kW, COP = 4.051
printed: 7.41, 1.83, 9.23, 4.06
CoolProp: QH = 9.233 kW, COP = 4.055


**Reading the output:** Q̇_L = 7.41 and Ẇ_in = 1.83 agree. Q̇_H = 9.236 (printed 9.23) and COP = 4.051 (printed 4.06) are off by 0.006 and 0.009 because the linearly interpolated h₂ is about 0.03 kJ/kg above the equation of state; CoolProp gives 9.233 and 4.055, closer to the printed answer. COP tolerance ± 0.015.

## 4. Real cycle: compressor efficiency and subcooling (Cengel Example 11–2, Problem 11–18)

**Question:** with real states (superheated vapor into the compressor, subcooled liquid out of the condenser, non-isentropic compression), what are the COP and η_c compared with the ideal cycle?

In [4]:
def real(P1, T1, P2, T2, T3):                      # MPa, C, MPa, C (compressor exit), C (condenser exit liquid)
    h1, s1 = sh(P1, T1, 'h'), sh(P1, T1, 's'); h2s = h_at_s(P2, s1); h2 = sh(P2, T2, 'h'); h3 = A11col(T3, 'hf')
    return dict(h1=h1, h2=h2, h2s=h2s, h3=h3, qL=h1 - h3, w=h2 - h1, cop=(h1 - h3) / (h2 - h1), eta=(h2s - h1) / (h2 - h1))
e = real(0.14, -10, 0.8, 50, 26); print(f"Example 11-2: h1 = {e['h1']:.2f}, h2 = {e['h2']:.2f}, h2s = {e['h2s']:.2f}, h3 = {e['h3']:.2f}")
print(f"  QL = {0.05*e['qL']:.2f} kW, Win = {0.05*e['w']:.2f} kW, eta_c = {e['eta']:.3f}, COP = {e['cop']:.2f} (ideal cycle: {c['cop']:.2f})")
g = real(0.14, -10, 0.7, 50, 24)
print(f"Problem 11-18: QL = {0.12*g['qL']:.2f} kW, Win = {0.12*g['w']:.2f} kW, eta_c = {g['eta']:.3f}, COP = {g['cop']:.2f}")
print('printed: 7.93, 2.02, 0.939, 3.93 and 19.4, 5.06, 0.825, 3.83')

Example 11-2: h1 = 246.36, h2 = 286.69, h2s = 284.21, h3 = 87.83
  QL = 7.93 kW, Win = 2.02 kW, eta_c = 0.938, COP = 3.93 (ideal cycle: 3.97)
Problem 11-18: QL = 19.37 kW, Win = 5.06 kW, eta_c = 0.826, COP = 3.83
printed: 7.93, 2.02, 0.939, 3.93 and 19.4, 5.06, 0.825, 3.83


**Reading the output:** Example 11–2: Q̇_L = 7.93 kW, Ẇ = 2.02 kW, η_c = 0.938, COP = 3.93, below the 3.97 of the ideal cycle despite the subcooling, because the work rises more than the cooling. Problem 11–18: η_c = 0.826 from the tables (printed 0.825; CoolProp 0.825): the linearly interpolated h₂ₛ is 0.04 kJ/kg high.

## 5. Versus Carnot, a turbine instead of the valve, and the 2–4 % per °C rule

**Question:** what fraction of the Carnot COP does the Tutorials I cycle reach? If the valve is replaced by an isentropic turbine, how much does the COP rise (Cengel Example 11–1, Discussion)? Is the "COP +2–4 % per °C of evaporating temperature" rule right?

In [5]:
TLk, THk = A12col(140, 'Tsat') + 273.15, A12col(800, 'Tsat') + 273.15
ccar = 1 / (THk / TLk - 1); print(f"Carnot between {TLk-273.15:.2f} C and {THk-273.15:.2f} C: COP = {ccar:.2f}; cycle/Carnot = {c['cop']/ccar:.2f}")
s3 = A12col(800, 'sf'); x4s = (s3 - A12col(140, 'sf')) / A12col(140, 'sfg'); h4s = A12col(140, 'hf') + x4s * A12col(140, 'hfg')
qLt = c['h1'] - h4s; wt = c['w'] - (c['h3'] - h4s)
print(f"turbine instead of valve: h4s = {h4s:.2f} kJ/kg (book 88.94), turbine power = {m*(c['h3']-h4s):.2f} kW, COP = {qLt/wt:.2f} (book 5.07), increase {100*(qLt/wt/c['cop']-1):.1f} %")
if HAVE_CP:
    def cop_T(Tl, Th):
        Pl = CP.PropsSI('P', 'T', Tl+273.15, 'Q', 1, 'R134a'); Ph = CP.PropsSI('P', 'T', Th+273.15, 'Q', 0, 'R134a')
        h1 = CP.PropsSI('H', 'P', Pl, 'Q', 1, 'R134a'); s1 = CP.PropsSI('S', 'P', Pl, 'Q', 1, 'R134a')
        return (h1 - CP.PropsSI('H', 'P', Ph, 'Q', 0, 'R134a')) / (CP.PropsSI('H', 'P', Ph, 'S', s1, 'R134a') - h1)
    for Tl in (-20, -10, 0):
        print(f"T_evap {Tl:4d} C, T_cond 40 C: COP = {cop_T(Tl, 40):.3f}; +1 C evaporating: {100*(cop_T(Tl+1, 40)/cop_T(Tl, 40)-1):.2f} %; -1 C condensing: {100*(cop_T(Tl, 39)/cop_T(Tl, 40)-1):.2f} %")

Carnot between -18.77 C and 31.31 C: COP = 5.08; cycle/Carnot = 0.78
turbine instead of valve: h4s = 88.94 kJ/kg (book 88.94), turbine power = 0.33 kW, COP = 5.06 (book 5.07), increase 27.5 %
T_evap  -20 C, T_cond 40 C: COP = 3.064; +1 C evaporating: 2.66 %; -1 C condensing: 2.54 %
T_evap  -10 C, T_cond 40 C: COP = 4.029; +1 C evaporating: 2.95 %; -1 C condensing: 2.83 %
T_evap    0 C, T_cond 40 C: COP = 5.494; +1 C evaporating: 3.43 %; -1 C condensing: 3.30 %


**Reading the output:** Carnot COP = 5.08; the ideal cycle = 3.97, i.e. 0.78 of Carnot. A turbine instead of the valve gives h₄ₛ = 88.94 kJ/kg and COP = 5.06: up 27.5 % (the book rounds to 5.07 and 28 %). Cengel's rule of thumb holds: each °C of higher evaporating or lower condensing temperature raises the COP by about 2.5–3.5 % at the test points.

## 6. Two-stage compression with a flash chamber (Cengel Problem 11–96, extension)

**Question:** between 0.8 and 0.14 MPa with a flash chamber at 0.4 MPa: what are the flashing fraction x₆, the heat removed and the work per kg through the condenser, and the COP (printed 0.165; 146.4; 32.6; 4.49)?

In [6]:
hf4, hg4 = A12col(400, 'hf'), A12col(400, 'hg'); h5 = A12col(800, 'hf'); x6 = (h5 - hf4) / (hg4 - hf4)       # flashing fraction
h1, s1 = A12col(140, 'hg'), A12col(140, 'sg'); h2 = h_at_s(0.4, s1)                  # low-pressure compressor to 0.4 MPa
h9 = (1 - x6) * h2 + x6 * hg4; s9 = (1 - x6) * s1 + x6 * A12col(400, 'sg'); h4 = h_at_s(0.8, s9)   # mixing, then high-pressure compressor
qL = (1 - x6) * (h1 - hf4); w = (1 - x6) * (h2 - h1) + (h4 - h9)
print(f"x6 = {x6:.3f}, qL = {qL:.1f} kJ/kg, w = {w:.1f} kJ/kg, COP = {qL/w:.2f}")
print(f"single stage 0.14-0.8 MPa: COP = {c['cop']:.2f}")

x6 = 0.165, qL = 146.4 kJ/kg, w = 32.6 kJ/kg, COP = 4.49
single stage 0.14-0.8 MPa: COP = 3.97


**Reading the output:** x₆ = 0.165; q_L = 146.4, w = 32.6 kJ/kg, COP = 4.49: matching the printed 0.165; 146.4; 32.6; 4.49. Compared with a single stage at the same pressure limits (COP 3.97), the two-stage system is 13 % higher.

## 7. Gas refrigeration: open aircraft cooling (Problem 11–97) and regeneration (Problem 11–61)

**Question:** air at 30 °C, 100 kPa is compressed to 250 kPa, cooled to 70 °C and expanded through a turbine: what is the cabin inlet temperature? Closed form versus numerical integration of the isentrope; does COP = 1/(r_p^((k−1)/k) − 1) hold?

In [7]:
k, cp = 1.4, 1.005
def T_isen(T1, p1, p2):                              # numerical isentrope: dT/dp = R T/(cp p), R = cp (k-1)/k
    R = cp * (k - 1) / k; return solve_ivp(lambda p, T: [R * T[0] / (cp * p)], (p1, p2), [T1], rtol=1e-12, atol=1e-12).y[0, -1]
x = 2.5 ** ((k - 1) / k); T1, T3 = 303.15, 343.15
T2 = T1 * x; T4 = T3 / x
print(f"Problem 11-97: T2 = {T2-273.15:.1f} C, T4 = {T4-273.15:.1f} C (printed -9 C); ODE: T4 = {T_isen(T3, 250, 100)-273.15:.1f} C")
q = cp * (T1 - T4); w = cp * (T2 - T1) - cp * (T3 - T4); print(f"COP = {q/w:.3f}; closed form 1/(x-1) = {1/(x-1):.3f}")
# regenerative cycle (Problem 11-61): T1 = -7 C, T3 = 27 C after the sink, T4 = -15 C after the regenerator, r_p = 4
x4 = 4 ** ((k - 1) / k); T1, T3, T4 = 266.15, 300.15, 258.15
T2, T5, T6 = T1 * x4, T4 / x4, T1 - (T3 - T4)
qL = cp * (T6 - T5); wn = cp * (T2 - T1) - cp * (T4 - T5)
print(f"Problem 11-61: lowest T = {T5-273.15:.1f} C, COP = {qL/wn:.2f}, mass flow for 12 kW = {12/qL:.3f} kg/s (printed -99.4 C, 1.12, 0.237)")
if HAVE_CP:
    A = lambda o, *a: CP.PropsSI(o, *a, 'Air')
    T4cp = A('T', 'P', 100e3, 'S', A('S', 'T', 343.15, 'P', 250e3)); print(f"CoolProp Air (real gas): T4 = {T4cp-273.15:.2f} C")

Problem 11-97: T2 = 120.7 C, T4 = -9.0 C (printed -9 C); ODE: T4 = -9.0 C
COP = 3.342; closed form 1/(x-1) = 3.342
Problem 11-61: lowest T = -99.4 C, COP = 1.12, mass flow for 12 kW = 0.237 kg/s (printed -99.4 C, 1.12, 0.237)
CoolProp Air (real gas): T4 = -9.09 C


**Reading the output:** T₂ = 120.7 °C, T₄ = -9.0 °C ≈ −9 °C (printed −9 °C); the numerical integration equals the closed form and CoolProp (real gas) is within 0.3 K. COP = 3.342 = 1/(x − 1): it depends only on r_p. Problem 11–61: lowest temperature -99.4 °C, COP = 1.12, ṁ = 0.237 kg/s: matching the printed answers.

## 8. Gas refrigeration with variable specific heats: Table A-17 (Cengel Problem 11–56)

**Question:** air enters the compressor at 12 °C (285 K), 50 kPa and the turbine at 47 °C (320 K), 250 kPa; 0.08 kg/s. What are the refrigeration rate, net power and COP from the P_r column of Table A-17 (printed 6.67 kW; 3.88 kW; 1.72), and how do they differ from constant c_p?

In [8]:
rp, T1, T3, m = 5.0, 285.0, 320.0, 0.08
T2 = inv('Pr', rp * tab('Pr', T1)); T4 = inv('Pr', tab('Pr', T3) / rp)                 # isentropic: Pr2/Pr1 = rp
h1, h2, h3, h4 = tab('h', T1), tab('h', T2), tab('h', T3), tab('h', T4)
qL = h1 - h4; wnet = (h2 - h1) - (h3 - h4)
print(f"T2 = {T2:.1f} K, T4 = {T4:.1f} K; qL = {qL:.2f} kJ/kg, wnet = {wnet:.2f} kJ/kg")
print(f"QL = {m*qL:.2f} kW, Wnet = {m*wnet:.2f} kW, COP = {qL/wnet:.2f} (printed 6.67, 3.88, 1.72)")
print(f"constant cp: COP = {1/(rp**(0.4/1.4)-1):.3f}")

T2 = 450.4 K, T4 = 201.8 K; qL = 83.38 kJ/kg, wnet = 48.50 kJ/kg
QL = 6.67 kW, Wnet = 3.88 kW, COP = 1.72 (printed 6.67, 3.88, 1.72)
constant cp: COP = 1.713


**Reading the output:** T₂ = 450.4 K, T₄ = 201.8 K (below 273 K, where the A-2c c_p(T) polynomial does not apply, so the check uses the integral of CoolProp's c_p, within 1 %); Q̇_L = 6.67 kW, Ẇ = 3.88 kW, COP = 1.72: matching the printed answers. With constant c_p the COP = 1.713: close but 0.4 % different.

## 9. Cooling pack: the slide temperature chain and unit conversions (illustration only)

**Question:** what temperature chain do the slide p.32–35 numbers give when added/subtracted? What are 30–40 psi and 40.000–50.000 rpm in SI units? If compression/expansion were isentropic, what pressure ratios would those temperature changes imply (the slides give NO pressure ratio)?

In [9]:
T0 = 150.0                                                  # bleed air entering the primary heat exchanger (slide p.32)
after_primary = T0 - 100                                      # "about 100 C" drop (p.32): derived value, not printed on the slide
after_comp = (after_primary + 30, after_primary + 40)         # compressor rise 30-40 C (p.33)
after_secondary = (after_comp[0] - 70, after_comp[1] - 60)    # secondary drop 60-70 C (p.33): lowest and highest possible
print(f"after primary exchanger: {after_primary:.0f} C; after ACM compressor: {after_comp[0]:.0f}-{after_comp[1]:.0f} C; after secondary: {after_secondary[0]:.0f}-{after_secondary[1]:.0f} C")
print("slide p.34 turbine inlet '20 to 10 C' (read as 10-20 C) lies inside the derived range" if after_secondary[0] <= 10 and 20 <= after_secondary[1] else "inconsistent")
psi = 6.894757
print(f"30-40 psi = {30*psi:.0f}-{40*psi:.0f} kPa; if gauge, absolute = {30*psi+101.325:.0f}-{40*psi+101.325:.0f} kPa")
print(f"40 000-50 000 rpm = {40000*2*math.pi/60:.0f}-{50000*2*math.pi/60:.0f} rad/s")
PRc = ((50 + 273.15 + 35) / (50 + 273.15)) ** 3.5; PRt = ((15 + 273.15) / (-10 + 273.15)) ** 3.5
print(f"illustration (isentropic, k = 1.4): +35 K from 50 C -> pressure ratio {PRc:.2f}; 15 C -> -10 C expansion -> pressure ratio {PRt:.2f}")

after primary exchanger: 50 C; after ACM compressor: 80-90 C; after secondary: 10-30 C
slide p.34 turbine inlet '20 to 10 C' (read as 10-20 C) lies inside the derived range
30-40 psi = 207-276 kPa; if gauge, absolute = 308-377 kPa
40 000-50 000 rpm = 4189-5236 rad/s
illustration (isentropic, k = 1.4): +35 K from 50 C -> pressure ratio 1.43; 15 C -> -10 C expansion -> pressure ratio 1.37


**Reading the output:** the chain 150 → 50 → 80–90 → 10–30 °C is consistent with the 10–20 °C of p.34 (the slide writes "20 to 10" in reverse). 30–40 psi = 207–276 kPa (if gauge, absolute 308–377 kPa: the slide does not say). 40.000–50.000 rpm = 4189–5236 rad/s. The illustrative pressure ratios 1.43 (compressor) and 1.37 (turbine) are inferences under an isentropic assumption, not slide data.